In [7]:
# strategy backtesting and P&L conversion

# this will be fully populated once the data (ML) team delivers the final probabilities
# might be adjusted based on structure but this should generally be ready
# also need to add the loading of the data files - this will be done when data is ready
# plotting will be done after as well (equity curve, drawdown curve) as well as the comparison table

# Assumptions :
# - transaction costs are 0.1%, slippage costs are 0.02% per trade
# - signals generated at the end of day (t) are executed at the close of day t, returns are realized t+1
# - missing prob signals are treated as 0. missing returns filled with 0

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [10]:
def compute_metrics(daily_perf, equity_curve, drawdown_curve, turnover_series):
    """Standard performance metrics annualized to 252 trading days."""
    
    days = len(daily_perf)
    if days == 0:
        return {}
    
    ann_factor = 252 
    
    cagr = (equity_curve.iloc[-1])**(ann_factor / days) - 1
    volatility = daily_perf.std() * np.sqrt(ann_factor)
    
    sharpe = (daily_perf.mean() / daily_perf.std()) * np.sqrt(ann_factor) if daily_perf.std() != 0 else 0
    
    downside_returns = daily_perf[daily_perf < 0]
    downside_vol = downside_returns.std() * np.sqrt(ann_factor)
    sortino = (daily_perf.mean() * ann_factor) / downside_vol if downside_vol != 0 else 0
    
    max_dd = drawdown_curve.min()
    annual_turnover = turnover_series.mean() * ann_factor
    
    return {
        'CAGR': cagr,
        'Volatility': volatility,
        'Sharpe': sharpe,
        'Sortino': sortino,
        'Max Drawdown': max_dd,
        'Annual Turnover': annual_turnover
    }

In [11]:
def run_strategy(df, prob_col, sizing_method='threshold', commission=0.001, slippage=0.0002):
    """
    Simulates portfolio performance based on probability signals.
    sizing_method: 'threshold' or 'continuous'
    """
    bt = df.copy().sort_values(['ticker', 'date'])

    # integrating signals
    if sizing_method == 'threshold':
        bt['target_weight'] = 0.0
        bt.loc[bt[prob_col] > 0.6, 'target_weight'] = 1.0
        bt.loc[bt[prob_col] < 0.4, 'target_weight'] = -1.0
    elif sizing_method == 'continuous':
        bt['target_weight'] = 2 * (bt[prob_col] - 0.5)
        bt['target_weight'] = bt['target_weight'].clip(lower=-1.0, upper=1.0)
    else:
        raise ValueError("sizing_method must be 'threshold' or 'continuous'")

    # shift weights by 1 to prevent lookahead bias 
    bt['actual_weight'] = bt.groupby('ticker')['target_weight'].shift(1).fillna(0)
    
    # turnover is abs diff in weight
    bt['turnover'] = bt.groupby('ticker')['actual_weight'].diff().abs().fillna(0)
    total_cost_rate = commission + slippage
    bt['costs'] = bt['turnover'] * total_cost_rate

    # calculated P&L
    bt['raw_return'] = bt['actual_weight'] * bt['return']
    bt['net_return'] = bt['raw_return'] - bt['costs']

    # output metrics, aggregated to portfolio level
    daily_perf = bt.groupby('date')['net_return'].mean()
    portfolio_turnover = bt.groupby('date')['turnover'].mean()

    equity_curve = (1 + daily_perf).cumprod()
    running_max = equity_curve.cummax()
    drawdown_curve = (equity_curve - running_max) / running_max

    metrics = compute_metrics(daily_perf, equity_curve, drawdown_curve, portfolio_turnover)

    return equity_curve, drawdown_curve, metrics

In [12]:
# run all four strategies through the exact same engine. this will work once all data is received
eq_mr, dd_mr, metrics_mr = run_strategy(mr_data, 'signal', 'threshold')
eq_mom, dd_mom, metrics_mom = run_strategy(mom_data, 'signal', 'threshold')
eq_ht, dd_ht, metrics_ht = run_strategy(features, 'tech_prob', 'continuous')
eq_hf, dd_hf, metrics_hf = run_strategy(features, 'full_prob', 'continuous')

NameError: name 'mr_data' is not defined